In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Ensure the base model is configured to output its hidden states
base_model.config.output_hidden_states = True

def get_last_token_embedding(text, current_model, current_tokenizer):
    # Tokenize the input text
    encoded = current_tokenizer(text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        # Run a forward pass requesting hidden states
        outputs = current_model(**encoded, output_hidden_states=True)
        
    # Extract the hidden states from the very last layer
    last_layer_hidden_states = outputs.hidden_states[-1]
    
    # Isolate the vector corresponding to the final token in the sequence
    final_token_embedding = last_layer_hidden_states[0, -1, :].cpu().numpy()
    return final_token_embedding

# Isolate unique labels from your test dataset to avoid redundant processing
unique_labels = df['completion'].dropna().unique().tolist()

before_embeddings = []
after_embeddings = []

print("\nExtracting embeddings before and after fine-tuning...")
for label in tqdm(unique_labels):
    
    # 1. Extract Base Model Embeddings (Adapter Disabled)
    with model.disable_adapter():
        emb_before = get_last_token_embedding(label, model, tokenizer)
        before_embeddings.append(emb_before)
        
    # 2. Extract Fine-Tuned Embeddings (Adapter Enabled automatically outside the context manager)
    emb_after = get_last_token_embedding(label, model, tokenizer)
    after_embeddings.append(emb_after)

before_embeddings = np.array(before_embeddings)
after_embeddings = np.array(after_embeddings)

# Combine both sets into a single array to establish a shared PCA space
all_embeddings = np.vstack((before_embeddings, after_embeddings))

print("Computing PCA...")
pca = PCA(n_components=2)
all_pca = pca.fit_transform(all_embeddings)

# Split the transformed data back into their respective groups
half_point = len(unique_labels)
before_pca = all_pca[:half_point]
after_pca = all_pca[half_point:]

## Plotting the Results
plt.figure(figsize=(12, 8))
plt.scatter(before_pca[:, 0], before_pca[:, 1], alpha=0.6, label='Base Model', color='tab:blue')
plt.scatter(after_pca[:, 0], after_pca[:, 1], alpha=0.6, label='Fine-Tuned', color='tab:red')

# Draw faint lines connecting the same label's coordinates to visualize the exact vector shift
for i in range(len(unique_labels)):
    plt.plot([before_pca[i, 0], after_pca[i, 0]], 
             [before_pca[i, 1], after_pca[i, 1]], 
             'k-', alpha=0.15)

plt.title("PCA of Label Embeddings: Base vs. Fine-Tuned", fontsize=14, fontweight='bold')
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

pca_chart_path = os.path.join(fig_dir, "pca_embeddings_shift.png")
plt.tight_layout()
plt.savefig(pca_chart_path, dpi=200)